# CST 627 Week 4 — Three Generative Recipes, One Colab Session
## DDPM · Rectified Flow · 1D Fourier Neural Operator

This notebook builds and measures three toy models on a single free-tier Colab GPU:

1. **DDPM** — epsilon-prediction diffusion on a self-generated two-moons dataset (cosine, variance-preserving schedule).
2. **Rectified Flow** — velocity-field flow matching on the *same* two-moons dataset, for a controlled comparison with DDPM.
3. **1D Fourier Neural Operator (FNO)** — surrogate for a self-written finite-difference heat-equation solver, trained on self-generated random smooth initial conditions.

The notebook also runs two **deliberately induced failures** (bad DDPM noise schedule, excessive learning rate), logs the evidence, repairs them, and ends with a measurement-based comparison and pilot recommendation.

**Rule for this notebook:** every number reported below is computed at run time from the cells above it. Nothing is hand-typed as a "typical" or "expected" value. If a cell hasn't been run, the print statements below it have not produced real numbers yet — rerun the notebook top to bottom in Colab (GPU runtime) before trusting any printed table.


## 0. Setup

In [ ]:
import math
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU for the measurements this "
          "assignment requires (CUDA timing / peak VRAM). The notebook will still run on CPU but VRAM "
          "figures will read 'N/A (CPU)'.")

# results dict: every measured section writes its numbers here, and nothing else in the
# notebook (failure log, final recommendation) is allowed to report a number that isn't in here.
results = {}


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5

def cuda_timer_start():
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    return time.perf_counter()

def cuda_timer_stop(t0):
    if device.type == "cuda":
        torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    else:
        peak_mem = float("nan")
    elapsed = time.perf_counter() - t0
    return elapsed, peak_mem


## 1. Self-generated dataset: two moons

Written from scratch (no external dataset download) so the "self-generated data scripts" requirement is
unambiguous: two half-circle arcs with additive Gaussian noise, standardized to zero mean / unit variance.


In [ ]:
def make_two_moons(n_samples=2000, noise=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n1 = n_samples // 2
    n2 = n_samples - n1
    theta1 = rng.uniform(0, np.pi, n1)
    theta2 = rng.uniform(0, np.pi, n2)
    moon1 = np.stack([np.cos(theta1), np.sin(theta1)], axis=1)
    moon2 = np.stack([1 - np.cos(theta2), 1 - np.sin(theta2) - 0.5], axis=1)
    X = np.concatenate([moon1, moon2], axis=0).astype(np.float64)
    X += rng.normal(scale=noise, size=X.shape)
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    perm = rng.permutation(n_samples)
    return X[perm].astype(np.float32)

N_DATA = 2000
data_np = make_two_moons(N_DATA, noise=0.05, seed=SEED)
data = torch.from_numpy(data_np).to(device)
print(f"Two-moons dataset: {data.shape[0]} points, dim={data.shape[1]}")

plt.figure(figsize=(4, 4))
plt.scatter(data_np[:, 0], data_np[:, 1], s=4, alpha=0.6)
plt.title("Self-generated two-moons data (n=2000)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.axis("equal")
plt.show()


## 2. Shared distributional quality metric: MMD (RBF kernel)

Both DDPM and rectified flow are scored with the **same** metric so the comparison in Section 6 is apples-to-apples:
squared Maximum Mean Discrepancy with a multi-bandwidth RBF kernel between generated samples and held-out real data.
Lower is better; MMD=0 only when the two empirical distributions match exactly (in the RKHS sense).


In [ ]:
def mmd_rbf(X, Y, sigmas=(0.5, 1.0, 2.0, 4.0, 8.0)):
    """Unbiased squared MMD, RBF kernel averaged over `sigmas`. X, Y: [n,d] and [m,d] tensors."""
    XX = torch.cdist(X, X) ** 2
    YY = torch.cdist(Y, Y) ** 2
    XY = torch.cdist(X, Y) ** 2
    def rbf_mix(D2):
        return sum(torch.exp(-D2 / (2 * s ** 2)) for s in sigmas) / len(sigmas)
    K_XX, K_YY, K_XY = rbf_mix(XX), rbf_mix(YY), rbf_mix(XY)
    m, n = X.shape[0], Y.shape[0]
    mmd2 = ((K_XX.sum() - K_XX.trace()) / (m * (m - 1))
            + (K_YY.sum() - K_YY.trace()) / (n * (n - 1))
            - 2 * K_XY.mean())
    return mmd2.item()

# held-out reference set for evaluation (re-generated with a different seed so it isn't the training set)
eval_ref_np = make_two_moons(1000, noise=0.05, seed=SEED + 999)
eval_ref = torch.from_numpy(eval_ref_np).to(device)
print("MMD reference set ready:", eval_ref.shape)


## 3. DDPM — epsilon-prediction diffusion (cosine / variance-preserving schedule)

- Closed-form forward corruption: `x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps`
- Small timestep-conditioned MLP predicts `eps`
- Cosine noise schedule (Nichol & Dhariwal, *Improved DDPM*)
- Ancestral sampling for `T` steps


In [ ]:
def cosine_alpha_bar(t_frac, s=0.008):
    return np.cos((t_frac + s) / (1 + s) * np.pi / 2) ** 2

def make_cosine_schedule(T, s=0.008):
    steps = np.arange(T + 1)
    f = cosine_alpha_bar(steps / T, s)
    alpha_bar = f / f[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    betas = np.clip(betas, 1e-8, 0.999)
    return betas.astype(np.float32)

def schedule_tensors(betas_np, device):
    betas = torch.from_numpy(betas_np).to(device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars

T_DDPM = 500
betas_cos_np = make_cosine_schedule(T_DDPM)
betas_cos, alphas_cos, alpha_bars_cos = schedule_tensors(betas_cos_np, device)
print(f"Cosine schedule: T={T_DDPM}, beta range [{betas_cos_np.min():.2e}, {betas_cos_np.max():.2e}], "
      f"alpha_bar range [{alpha_bars_cos[-1].item():.2e}, {alpha_bars_cos[0].item():.2e}]")

plt.figure(figsize=(4, 3))
plt.plot(alpha_bars_cos.cpu().numpy())
plt.title("Cosine schedule: alpha_bar(t)")
plt.xlabel("t"); plt.ylabel("alpha_bar")
plt.show()


In [ ]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TimeConditionedMLP(nn.Module):
    """Shared architecture for both DDPM (integer t in [0,T)) and rectified flow (continuous t in [0,1])."""
    def __init__(self, dim=2, hidden=128, time_dim=32, time_scale=1.0):
        super().__init__()
        self.time_scale = time_scale
        self.time_embed = SinusoidalEmbedding(time_dim)
        self.net = nn.Sequential(
            nn.Linear(dim + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):
        temb = self.time_embed(t * self.time_scale)
        return self.net(torch.cat([x, temb], dim=-1))


In [ ]:
def train_ddpm(data, betas, alphas, alpha_bars, n_steps=3000, batch_size=256, lr=1e-3,
                clip_grad_norm=None, log_every=50, seed=SEED):
    torch.manual_seed(seed)
    T = betas.shape[0]
    model = TimeConditionedMLP(dim=2, hidden=128, time_dim=32, time_scale=1.0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loss_history, grad_norm_history, step_log = [], [], []
    diverged = False
    t0 = cuda_timer_start()
    n_data = data.shape[0]
    for step in range(n_steps):
        idx = torch.randint(0, n_data, (batch_size,), device=device)
        x0 = data[idx]
        t = torch.randint(0, T, (batch_size,), device=device)
        eps = torch.randn_like(x0)
        sqrt_ab = torch.sqrt(alpha_bars[t])[:, None]
        sqrt_1mab = torch.sqrt(1 - alpha_bars[t])[:, None]
        x_t = sqrt_ab * x0 + sqrt_1mab * eps

        eps_pred = model(x_t, t)
        loss = F.mse_loss(eps_pred, eps)

        opt.zero_grad()
        loss.backward()
        gn = grad_norm(model)
        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

        if not torch.isfinite(loss):
            diverged = True
            loss_history.append(float("nan"))
            grad_norm_history.append(gn)
            step_log.append(step)
            print(f"[train_ddpm] step {step}: loss is {loss.item()} -> stopping (NaN/Inf).")
            break

        opt.step()

        if step % log_every == 0 or step == n_steps - 1:
            loss_history.append(loss.item())
            grad_norm_history.append(gn)
            step_log.append(step)

    train_time, peak_mem = cuda_timer_stop(t0)
    return {
        "model": model, "loss_history": loss_history, "grad_norm_history": grad_norm_history,
        "step_log": step_log, "train_time_s": train_time, "peak_train_mem_mb": peak_mem,
        "diverged": diverged, "n_params": count_params(model), "T": T,
    }


@torch.no_grad()
def ddpm_sample(model, n_samples, betas, alphas, alpha_bars, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    T = betas.shape[0]
    x = torch.randn(n_samples, 2, device=device)
    t0 = cuda_timer_start()
    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch)
        alpha_t, alpha_bar_t, beta_t = alphas[t], alpha_bars[t], betas[t]
        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)
        mean = coef1 * (x - coef2 * eps_pred)
        if t > 0:
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mean
    sample_time, peak_mem = cuda_timer_stop(t0)
    return x, {"sample_time_s": sample_time, "peak_sample_mem_mb": peak_mem, "nfe": T}


In [ ]:
ddpm_run = train_ddpm(data, betas_cos, alphas_cos, alpha_bars_cos, n_steps=3000, batch_size=256, lr=1e-3)
print(f"DDPM (cosine, baseline) params={ddpm_run['n_params']}, "
      f"train_time={ddpm_run['train_time_s']:.2f}s, peak_train_mem={ddpm_run['peak_train_mem_mb']:.1f}MB, "
      f"final_loss={ddpm_run['loss_history'][-1]:.4f}, diverged={ddpm_run['diverged']}")

plt.figure(figsize=(4, 3))
plt.plot(ddpm_run["step_log"], ddpm_run["loss_history"])
plt.title("DDPM training loss (cosine schedule)")
plt.xlabel("step"); plt.ylabel("MSE(eps_pred, eps)")
plt.show()


In [ ]:
ddpm_samples, ddpm_sample_stats = ddpm_sample(ddpm_run["model"], 1000, betas_cos, alphas_cos, alpha_bars_cos, seed=SEED)
ddpm_mmd = mmd_rbf(ddpm_samples, eval_ref)

print(f"DDPM sampling: {ddpm_sample_stats['nfe']} NFEs, "
      f"latency={ddpm_sample_stats['sample_time_s']:.3f}s, "
      f"peak_sample_mem={ddpm_sample_stats['peak_sample_mem_mb']:.1f}MB")
print(f"DDPM MMD^2 vs held-out reference: {ddpm_mmd:.5f}")

results["ddpm_baseline"] = {
    "schedule": "cosine", "T": T_DDPM, "n_params": ddpm_run["n_params"],
    "final_train_loss": ddpm_run["loss_history"][-1],
    "train_time_s": ddpm_run["train_time_s"], "peak_train_mem_mb": ddpm_run["peak_train_mem_mb"],
    "sample_time_s": ddpm_sample_stats["sample_time_s"],
    "peak_sample_mem_mb": ddpm_sample_stats["peak_sample_mem_mb"],
    "nfe": ddpm_sample_stats["nfe"], "mmd2": ddpm_mmd,
}

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=4, alpha=0.5)
axes[0].set_title("Real data"); axes[0].axis("equal")
axes[1].scatter(ddpm_samples.cpu().numpy()[:, 0], ddpm_samples.cpu().numpy()[:, 1], s=4, alpha=0.5, color="C1")
axes[1].set_title(f"DDPM samples (MMD^2={ddpm_mmd:.4f})"); axes[1].axis("equal")
plt.tight_layout(); plt.show()


## 4. Rectified Flow — velocity regression on the same two-moons dataset

Same architecture family and hidden width as the DDPM MLP (`TimeConditionedMLP`) for a controlled comparison.

- Straight-line interpolation: `x_t = (1-t) * x0 + t * x1`, `x0 ~ N(0,I)` noise, `x1` = data
- Target velocity: `x1 - x0`
- ODE sampling via Euler integration from `t=0` (noise) to `t=1` (data)


In [ ]:
def train_flow(data, n_steps=3000, batch_size=256, lr=1e-3, clip_grad_norm=None, log_every=50, seed=SEED):
    torch.manual_seed(seed)
    model = TimeConditionedMLP(dim=2, hidden=128, time_dim=32, time_scale=1.0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    loss_history, grad_norm_history, step_log = [], [], []
    diverged = False
    t0 = cuda_timer_start()
    n_data = data.shape[0]
    for step in range(n_steps):
        idx = torch.randint(0, n_data, (batch_size,), device=device)
        x1 = data[idx]
        x0 = torch.randn_like(x1)
        t = torch.rand(batch_size, device=device)
        x_t = (1 - t)[:, None] * x0 + t[:, None] * x1
        v_target = x1 - x0

        v_pred = model(x_t, t)
        loss = F.mse_loss(v_pred, v_target)

        opt.zero_grad()
        loss.backward()
        gn = grad_norm(model)
        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

        if not torch.isfinite(loss):
            diverged = True
            loss_history.append(float("nan"))
            grad_norm_history.append(gn)
            step_log.append(step)
            print(f"[train_flow] step {step}: loss is {loss.item()} -> stopping (NaN/Inf).")
            break

        opt.step()

        if step % log_every == 0 or step == n_steps - 1:
            loss_history.append(loss.item())
            grad_norm_history.append(gn)
            step_log.append(step)

    train_time, peak_mem = cuda_timer_stop(t0)
    return {
        "model": model, "loss_history": loss_history, "grad_norm_history": grad_norm_history,
        "step_log": step_log, "train_time_s": train_time, "peak_train_mem_mb": peak_mem,
        "diverged": diverged, "n_params": count_params(model),
    }


@torch.no_grad()
def flow_sample(model, n_samples, n_ode_steps=100, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    x = torch.randn(n_samples, 2, device=device)
    dt = 1.0 / n_ode_steps
    t0 = cuda_timer_start()
    for i in range(n_ode_steps):
        t = torch.full((n_samples,), i * dt, device=device)
        v = model(x, t)
        x = x + v * dt
    sample_time, peak_mem = cuda_timer_stop(t0)
    return x, {"sample_time_s": sample_time, "peak_sample_mem_mb": peak_mem, "nfe": n_ode_steps}


In [ ]:
flow_run = train_flow(data, n_steps=3000, batch_size=256, lr=1e-3)
print(f"Flow params={flow_run['n_params']}, train_time={flow_run['train_time_s']:.2f}s, "
      f"peak_train_mem={flow_run['peak_train_mem_mb']:.1f}MB, final_loss={flow_run['loss_history'][-1]:.4f}")

plt.figure(figsize=(4, 3))
plt.plot(flow_run["step_log"], flow_run["loss_history"])
plt.title("Rectified flow training loss")
plt.xlabel("step"); plt.ylabel("MSE(v_pred, v_target)")
plt.show()


In [ ]:
N_ODE_STEPS = 100
flow_samples, flow_sample_stats = flow_sample(flow_run["model"], 1000, n_ode_steps=N_ODE_STEPS, seed=SEED)
flow_mmd = mmd_rbf(flow_samples, eval_ref)

print(f"Flow sampling: {flow_sample_stats['nfe']} NFEs, "
      f"latency={flow_sample_stats['sample_time_s']:.3f}s, "
      f"peak_sample_mem={flow_sample_stats['peak_sample_mem_mb']:.1f}MB")
print(f"Flow MMD^2 vs held-out reference: {flow_mmd:.5f}")

results["flow_baseline"] = {
    "n_params": flow_run["n_params"], "final_train_loss": flow_run["loss_history"][-1],
    "train_time_s": flow_run["train_time_s"], "peak_train_mem_mb": flow_run["peak_train_mem_mb"],
    "sample_time_s": flow_sample_stats["sample_time_s"],
    "peak_sample_mem_mb": flow_sample_stats["peak_sample_mem_mb"],
    "nfe": flow_sample_stats["nfe"], "mmd2": flow_mmd,
}

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=4, alpha=0.5)
axes[0].set_title("Real data"); axes[0].axis("equal")
axes[1].scatter(flow_samples.cpu().numpy()[:, 0], flow_samples.cpu().numpy()[:, 1], s=4, alpha=0.5, color="C2")
axes[1].set_title(f"Flow samples (MMD^2={flow_mmd:.4f})"); axes[1].axis("equal")
plt.tight_layout(); plt.show()


## 5. DDPM vs Rectified Flow — controlled comparison

Same dataset, same MLP capacity, same evaluation metric (MMD² against a held-out reference set). The table below
is built entirely from `results[...]` populated above — nothing here is typed by hand.


In [ ]:
def print_ddpm_flow_table(results):
    rows = [
        ("model params", "n_params", "{:d}"),
        ("final train loss", "final_train_loss", "{:.4f}"),
        ("train wall-clock (s)", "train_time_s", "{:.2f}"),
        ("peak train VRAM (MB)", "peak_train_mem_mb", "{:.1f}"),
        ("sampling latency (s)", "sample_time_s", "{:.3f}"),
        ("peak sampling VRAM (MB)", "peak_sample_mem_mb", "{:.1f}"),
        ("sampling NFEs", "nfe", "{:d}"),
        ("MMD^2 vs held-out data (lower=better)", "mmd2", "{:.5f}"),
    ]
    d, f = results["ddpm_baseline"], results["flow_baseline"]
    print(f"{'metric':35s} {'DDPM (cosine)':>18s} {'Rectified Flow':>18s}")
    print("-" * 73)
    for label, key, fmt in rows:
        dv, fv = d[key], f[key]
        print(f"{label:35s} {fmt.format(dv):>18s} {fmt.format(fv):>18s}")

print_ddpm_flow_table(results)


## 6. Neural Operator — 1D FNO on self-generated heat-equation data

### 6.1 Self-written finite-difference heat solver

Explicit FTCS scheme for `u_t = alpha * u_xx` on `[0, L]` with Dirichlet BCs `u(0,t)=u(L,t)=0`, stability
enforced via `r = alpha*dt/dx^2 <= 0.5`.


In [ ]:
L_DOMAIN = 1.0
ALPHA_HEAT = 0.05

def fd_heat_solver(u0, alpha, dx, dt, n_steps):
    """u0: [B, N] (Dirichlet-consistent, i.e. u0[:,0]=u0[:,-1]=0). Returns u after n_steps."""
    r = alpha * dt / dx ** 2
    assert r <= 0.5, f"FTCS unstable: r={r:.3f} > 0.5 (reduce dt or increase dx)"
    u = u0.copy()
    for _ in range(n_steps):
        u_new = u.copy()
        u_new[:, 1:-1] = u[:, 1:-1] + r * (u[:, 2:] - 2 * u[:, 1:-1] + u[:, :-2])
        u_new[:, 0] = 0.0
        u_new[:, -1] = 0.0
        u = u_new
    return u

def make_grid(n_points):
    return np.linspace(0, L_DOMAIN, n_points)


### 6.2 Validate the solver against a known analytic solution

Initial condition `u0(x) = sin(pi x / L)` has closed-form solution
`u(x,t) = exp(-alpha (pi/L)^2 t) sin(pi x / L)` under this BC. We solve numerically and compare, and only
proceed to generate training labels if the numerical solver's relative L2 error against the analytic solution
is small.


In [ ]:
def relative_l2(pred, true, axis=-1):
    num = np.linalg.norm(pred - true, axis=axis)
    den = np.linalg.norm(true, axis=axis) + 1e-12
    return num / den

N_VALIDATE = 64
x_val = make_grid(N_VALIDATE)
dx_val = x_val[1] - x_val[0]
dt_val = 0.4 * dx_val ** 2 / ALPHA_HEAT  # r=0.4, safely under the 0.5 stability bound
T_FINAL = 0.1
n_steps_val = int(round(T_FINAL / dt_val))

u0_val = np.sin(np.pi * x_val / L_DOMAIN)[None, :]
u_numeric = fd_heat_solver(u0_val, ALPHA_HEAT, dx_val, dt_val, n_steps_val)[0]
u_analytic = np.exp(-ALPHA_HEAT * (np.pi / L_DOMAIN) ** 2 * (n_steps_val * dt_val)) * np.sin(np.pi * x_val / L_DOMAIN)

solver_rel_l2 = relative_l2(u_numeric, u_analytic)
print(f"Solver validation: r={ALPHA_HEAT*dt_val/dx_val**2:.3f}, n_steps={n_steps_val}, "
      f"relative L2 vs analytic solution = {solver_rel_l2:.2e}")
assert solver_rel_l2 < 1e-2, "Finite-difference solver does not match the analytic solution closely enough; do not trust generated labels."

plt.figure(figsize=(4, 3))
plt.plot(x_val, u0_val[0], label="u0 (t=0)")
plt.plot(x_val, u_analytic, "--", label="analytic u(x,T)")
plt.plot(x_val, u_numeric, ":", label="FD solver u(x,T)")
plt.legend(); plt.title(f"Solver validation (rel. L2={solver_rel_l2:.1e})")
plt.xlabel("x"); plt.ylabel("u")
plt.show()


### 6.3 Self-generated dataset: random smooth initial conditions -> solved output

Random truncated sine series (automatically satisfies the Dirichlet BC) with coefficients decaying in the mode
number so the initial conditions stay smooth.


In [ ]:
def random_smooth_ic(x, n_modes=5, seed=None):
    rng = np.random.default_rng(seed)
    u0 = np.zeros_like(x)
    for k in range(1, n_modes + 1):
        coef = rng.normal(scale=1.0 / k ** 1.5)
        u0 += coef * np.sin(k * np.pi * x / L_DOMAIN)
    return u0

N_TRAIN_GRID = 64
x_train_grid = make_grid(N_TRAIN_GRID)
dx_train = x_train_grid[1] - x_train_grid[0]
dt_train = 0.4 * dx_train ** 2 / ALPHA_HEAT
n_steps_train = int(round(T_FINAL / dt_train))

N_FNO_SAMPLES = 2000
u0_all = np.stack([random_smooth_ic(x_train_grid, n_modes=5, seed=1000 + i) for i in range(N_FNO_SAMPLES)])
u_final_all = fd_heat_solver(u0_all, ALPHA_HEAT, dx_train, dt_train, n_steps_train)

n_train = int(0.8 * N_FNO_SAMPLES)
u0_train, u0_test = u0_all[:n_train], u0_all[n_train:]
uf_train, uf_test = u_final_all[:n_train], u_final_all[n_train:]
print(f"FNO dataset: grid={N_TRAIN_GRID} points, n_steps={n_steps_train}, "
      f"train={u0_train.shape[0]}, test={u0_test.shape[0]}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.plot(x_train_grid, u0_all[i], label="u0")
    ax.plot(x_train_grid, u_final_all[i], label=f"u(T={T_FINAL})")
    ax.set_title(f"sample {i}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


### 6.4 Small 1D FNO

Spectral convolution layers (FFT, truncated to `modes` low frequencies) — the standard Li et al. (2021)
FNO block, kept small (`width=32`, `modes=16`, `depth=4`) for toy-scale training.


In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        scale = 1.0 / (in_ch * out_ch)
        self.weight = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, C, N = x.shape
        x_ft = torch.fft.rfft(x, dim=-1)
        modes = min(self.modes, x_ft.shape[-1])
        out_ft = torch.zeros(B, self.out_ch, x_ft.shape[-1], dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :modes] = torch.einsum("bix,iox->box", x_ft[:, :, :modes], self.weight[:, :, :modes])
        return torch.fft.irfft(out_ft, n=N, dim=-1)


class FNOBlock(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.pointwise = nn.Conv1d(width, width, 1)

    def forward(self, x):
        return F.gelu(self.spectral(x) + self.pointwise(x))


class FNO1d(nn.Module):
    def __init__(self, modes=16, width=32, depth=4):
        super().__init__()
        self.fc_in = nn.Linear(2, width)  # (u0 value, x coordinate) per grid point
        self.blocks = nn.ModuleList([FNOBlock(width, modes) for _ in range(depth)])
        self.fc_out1 = nn.Linear(width, width)
        self.fc_out2 = nn.Linear(width, 1)

    def forward(self, u0, grid):
        x = torch.stack([u0, grid], dim=-1)      # [B,N,2]
        x = self.fc_in(x).permute(0, 2, 1)        # [B,width,N]
        for block in self.blocks:
            x = block(x)
        x = x.permute(0, 2, 1)                    # [B,N,width]
        x = F.gelu(self.fc_out1(x))
        return self.fc_out2(x).squeeze(-1)         # [B,N]


In [ ]:
def relative_l2_torch(pred, true):
    num = torch.norm(pred - true, dim=-1)
    den = torch.norm(true, dim=-1) + 1e-12
    return (num / den)

def train_fno(u0_train, uf_train, grid, n_epochs=200, batch_size=64, lr=1e-3, seed=SEED):
    torch.manual_seed(seed)
    model = FNO1d(modes=16, width=32, depth=4).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    u0_t = torch.from_numpy(u0_train).float().to(device)
    uf_t = torch.from_numpy(uf_train).float().to(device)
    grid_t = torch.from_numpy(grid).float().to(device).unsqueeze(0).expand(u0_t.shape[0], -1)

    n = u0_t.shape[0]
    loss_history = []
    t0 = cuda_timer_start()
    for epoch in range(n_epochs):
        perm = torch.randperm(n, device=device)
        epoch_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            pred = model(u0_t[idx], grid_t[idx])
            loss = F.mse_loss(pred, uf_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * idx.shape[0]
        loss_history.append(epoch_loss / n)
    train_time, peak_mem = cuda_timer_stop(t0)
    return {"model": model, "loss_history": loss_history, "train_time_s": train_time,
            "peak_train_mem_mb": peak_mem, "n_params": count_params(model)}


@torch.no_grad()
def eval_fno(model, u0_np, uf_np, grid_np):
    model.eval()
    u0_t = torch.from_numpy(u0_np).float().to(device)
    uf_t = torch.from_numpy(uf_np).float().to(device)
    grid_t = torch.from_numpy(grid_np).float().to(device).unsqueeze(0).expand(u0_t.shape[0], -1)
    t0 = cuda_timer_start()
    pred = model(u0_t, grid_t)
    infer_time, peak_mem = cuda_timer_stop(t0)
    rel_l2 = relative_l2_torch(pred, uf_t)
    model.train()
    return {"pred": pred.cpu().numpy(), "rel_l2_mean": rel_l2.mean().item(),
            "rel_l2_std": rel_l2.std().item(), "infer_time_s": infer_time, "peak_infer_mem_mb": peak_mem}


In [ ]:
fno_run = train_fno(u0_train, uf_train, x_train_grid, n_epochs=200, batch_size=64, lr=1e-3)
print(f"FNO params={fno_run['n_params']}, train_time={fno_run['train_time_s']:.2f}s, "
      f"peak_train_mem={fno_run['peak_train_mem_mb']:.1f}MB, final_loss={fno_run['loss_history'][-1]:.6f}")

plt.figure(figsize=(4, 3))
plt.plot(fno_run["loss_history"])
plt.yscale("log")
plt.title("FNO training loss (MSE)")
plt.xlabel("epoch"); plt.ylabel("MSE (log scale)")
plt.show()


### 6.5 Evaluate at training resolution and at a finer grid

The FNO is never retrained for the finer grid. We regenerate an independent test set directly at the finer
resolution (new random ICs, solved with the finite-difference solver at that resolution), run the *same*
trained model on it, and report the **measured** relative L2 — we do not claim discretization invariance,
we measure whether it transfers.


In [ ]:
# training-resolution test set (already held out above)
eval_train_res = eval_fno(fno_run["model"], u0_test, uf_test, x_train_grid)
print(f"FNO @ training resolution ({N_TRAIN_GRID} pts): "
      f"relative L2 = {eval_train_res['rel_l2_mean']:.4f} +/- {eval_train_res['rel_l2_std']:.4f}, "
      f"inference latency = {eval_train_res['infer_time_s']:.4f}s "
      f"({u0_test.shape[0]} samples), peak_mem={eval_train_res['peak_infer_mem_mb']:.1f}MB")

# finer-grid, independently generated test set
N_FINE_GRID = 128
x_fine_grid = make_grid(N_FINE_GRID)
dx_fine = x_fine_grid[1] - x_fine_grid[0]
dt_fine = 0.4 * dx_fine ** 2 / ALPHA_HEAT
n_steps_fine = int(round(T_FINAL / dt_fine))

N_FINE_TEST = 200
u0_fine = np.stack([random_smooth_ic(x_fine_grid, n_modes=5, seed=5000 + i) for i in range(N_FINE_TEST)])
uf_fine = fd_heat_solver(u0_fine, ALPHA_HEAT, dx_fine, dt_fine, n_steps_fine)

eval_fine_res = eval_fno(fno_run["model"], u0_fine, uf_fine, x_fine_grid)
print(f"FNO @ finer grid ({N_FINE_GRID} pts, zero-shot, no retraining): "
      f"relative L2 = {eval_fine_res['rel_l2_mean']:.4f} +/- {eval_fine_res['rel_l2_std']:.4f}, "
      f"inference latency = {eval_fine_res['infer_time_s']:.4f}s "
      f"({u0_fine.shape[0]} samples), peak_mem={eval_fine_res['peak_infer_mem_mb']:.1f}MB")

results["fno"] = {
    "n_params": fno_run["n_params"], "final_train_loss": fno_run["loss_history"][-1],
    "train_time_s": fno_run["train_time_s"], "peak_train_mem_mb": fno_run["peak_train_mem_mb"],
    "rel_l2_train_res": eval_train_res["rel_l2_mean"], "rel_l2_train_res_std": eval_train_res["rel_l2_std"],
    "infer_time_train_res_s": eval_train_res["infer_time_s"],
    "rel_l2_fine_res": eval_fine_res["rel_l2_mean"], "rel_l2_fine_res_std": eval_fine_res["rel_l2_std"],
    "infer_time_fine_res_s": eval_fine_res["infer_time_s"],
    "train_grid": N_TRAIN_GRID, "fine_grid": N_FINE_GRID,
}


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, ax in enumerate(axes):
    ax.plot(x_train_grid, uf_test[i], label="ground truth (solver)")
    ax.plot(x_train_grid, eval_train_res["pred"][i], "--", label="FNO prediction")
    ax.set_title(f"test sample {i} (train res.)"); ax.legend(fontsize=7)
plt.suptitle(f"FNO prediction vs ground truth @ training resolution "
             f"(mean rel. L2={eval_train_res['rel_l2_mean']:.4f})")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, ax in enumerate(axes):
    ax.plot(x_fine_grid, uf_fine[i], label="ground truth (solver)")
    ax.plot(x_fine_grid, eval_fine_res["pred"][i], "--", label="FNO prediction")
    ax.set_title(f"fine-grid sample {i}"); ax.legend(fontsize=7)
plt.suptitle(f"FNO prediction vs ground truth @ finer grid, zero-shot "
             f"(mean rel. L2={eval_fine_res['rel_l2_mean']:.4f})")
plt.tight_layout(); plt.show()

print("Note: FNO relative L2 and the DDPM/flow MMD^2 are not comparable numbers — they score different "
      "tasks (function-to-function PDE regression vs. distributional generative sample quality). "
      "Do not rank them against each other.")


## 7. Failure Experiment 1 — poor/mismatched DDPM noise schedule

**Setting:** replace the cosine schedule with a linear schedule whose `beta_end` is far outside the usual
DDPM range (0.8 instead of ~0.02), at the same `T=500`. This destroys the signal far too quickly: `alpha_bar`
collapses toward 0 well before `t=T`, so most of the reverse trajectory operates at a near-vanishing
signal-to-noise ratio.


In [ ]:
def make_linear_schedule(T, beta_start, beta_end):
    return np.linspace(beta_start, beta_end, T, dtype=np.float32)

BAD_BETA_START, BAD_BETA_END = 1e-4, 0.8
betas_bad_np = make_linear_schedule(T_DDPM, BAD_BETA_START, BAD_BETA_END)
betas_bad, alphas_bad, alpha_bars_bad = schedule_tensors(betas_bad_np, device)

plt.figure(figsize=(4, 3))
plt.plot(alpha_bars_cos.cpu().numpy(), label="cosine (baseline)")
plt.plot(alpha_bars_bad.cpu().numpy(), label=f"linear beta_end={BAD_BETA_END} (failure 1)")
plt.legend(); plt.title("alpha_bar(t): baseline vs mismatched schedule")
plt.xlabel("t"); plt.ylabel("alpha_bar")
plt.show()

# evidence: how early does alpha_bar collapse to near-zero SNR? (directly readable off the plot above)
def alpha_bar_collapse_step(alpha_bar_np, threshold=1e-2):
    below = np.where(alpha_bar_np < threshold)[0]
    return int(below[0]) if len(below) > 0 else None

collapse_cos = alpha_bar_collapse_step(alpha_bars_cos.cpu().numpy(), threshold=1e-2)
collapse_bad = alpha_bar_collapse_step(alpha_bars_bad.cpu().numpy(), threshold=1e-2)
frac_wasted_bad = (T_DDPM - collapse_bad) / T_DDPM
frac_wasted_cos = (T_DDPM - collapse_cos) / T_DDPM
print(f"alpha_bar first drops below 0.01 at t={collapse_cos} (cosine) vs t={collapse_bad} (mismatched), out of T={T_DDPM}")
print(f"Fraction of the schedule spent at near-zero SNR: cosine={frac_wasted_cos:.1%}, mismatched={frac_wasted_bad:.1%}")


In [ ]:
ddpm_bad_run = train_ddpm(data, betas_bad, alphas_bad, alpha_bars_bad, n_steps=3000, batch_size=256, lr=1e-3)
print(f"[Failure 1] final train loss={ddpm_bad_run['loss_history'][-1]:.4f} "
      f"(baseline cosine final loss={ddpm_run['loss_history'][-1]:.4f}), diverged={ddpm_bad_run['diverged']}")

plt.figure(figsize=(4, 3))
plt.plot(ddpm_run["step_log"], ddpm_run["loss_history"], label="cosine (baseline)")
plt.plot(ddpm_bad_run["step_log"], ddpm_bad_run["loss_history"], label="mismatched linear")
plt.legend(); plt.title("Failure 1: training loss, baseline vs mismatched schedule")
plt.xlabel("step"); plt.ylabel("loss")
plt.show()


In [ ]:
ddpm_bad_samples, ddpm_bad_sample_stats = ddpm_sample(ddpm_bad_run["model"], 1000, betas_bad, alphas_bad, alpha_bars_bad, seed=SEED)
ddpm_bad_mmd = mmd_rbf(ddpm_bad_samples, eval_ref)
n_nonfinite = (~torch.isfinite(ddpm_bad_samples)).sum().item()
print(f"[Failure 1] MMD^2 = {ddpm_bad_mmd:.5f} (baseline cosine MMD^2 = {ddpm_mmd:.5f}), "
      f"non-finite sample entries = {n_nonfinite}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].scatter(ddpm_samples.cpu().numpy()[:, 0], ddpm_samples.cpu().numpy()[:, 1], s=4, alpha=0.5, color="C1")
axes[0].set_title(f"Baseline cosine (MMD^2={ddpm_mmd:.4f})"); axes[0].axis("equal")
bad_np = torch.nan_to_num(ddpm_bad_samples, nan=0.0, posinf=0.0, neginf=0.0).cpu().numpy()
axes[1].scatter(bad_np[:, 0], bad_np[:, 1], s=4, alpha=0.5, color="C3")
axes[1].set_title(f"Failure 1: mismatched schedule (MMD^2={ddpm_bad_mmd:.4f})"); axes[1].axis("equal")
plt.tight_layout(); plt.show()

results["failure1_bad"] = {
    "beta_start": BAD_BETA_START, "beta_end": BAD_BETA_END, "T": T_DDPM,
    "final_train_loss": ddpm_bad_run["loss_history"][-1], "mmd2": ddpm_bad_mmd,
    "n_nonfinite_samples": n_nonfinite,
    "collapse_step_bad": collapse_bad, "collapse_step_cos": collapse_cos,
    "frac_near_zero_snr_bad": frac_wasted_bad, "frac_near_zero_snr_cos": frac_wasted_cos,
}


**Diagnosis:** the loss curve above looks broadly comparable to the cosine baseline — often even *lower*
— so loss alone would *not* catch this failure (predicting `eps` stays an easy regression problem when the
schedule destroys the signal early, since the training target is close to pure Gaussian noise for most of the
range). The evidence is in the `alpha_bar` schedule itself: under the mismatched schedule `alpha_bar` drops
below 0.01 by roughly `t=collapse_step_bad` out of `T`, versus only the last few steps for the cosine baseline
(`t=collapse_step_cos`) — printed and stored in `results["failure1_bad"]` above. That means the large majority
of the reverse-sampling trajectory runs at near-zero signal-to-noise ratio under the bad schedule, instead of
only the final handful of steps under cosine. Training barely notices (the regression target is trivially easy
in that regime), but the reverse process spends most of its budget denoising from an already-uninformative
state, and the generated samples end up off the data manifold — visible in the MMD² increase and the scatter
plot above relative to the cosine baseline.

**Repair:** revert to the cosine schedule (Section 3) — no architecture or training-loop change needed. The
"post-repair" result is exactly the `ddpm_baseline` entry in `results` from Section 3, since that run used
the identical architecture, step count and batch size and only differs in the noise schedule.


## 8. Failure Experiment 2 — excessive learning rate

**Setting:** retrain the same DDPM architecture and cosine schedule, but with `lr=1.0` instead of the
working `lr=1e-3` (three orders of magnitude higher). Track the loss curve, gradient norms, and explicit
NaN/Inf checks each step.


In [ ]:
BAD_LR = 1.0
ddpm_lr_run = train_ddpm(data, betas_cos, alphas_cos, alpha_bars_cos, n_steps=3000, batch_size=256, lr=BAD_LR)
print(f"[Failure 2] lr={BAD_LR}: stopped at step {ddpm_lr_run['step_log'][-1]}, "
      f"diverged={ddpm_lr_run['diverged']}, last logged loss={ddpm_lr_run['loss_history'][-1]}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(ddpm_run["step_log"], ddpm_run["loss_history"], label="lr=1e-3 (baseline)")
axes[0].plot(ddpm_lr_run["step_log"], ddpm_lr_run["loss_history"], label=f"lr={BAD_LR} (failure 2)")
axes[0].legend(); axes[0].set_title("Loss"); axes[0].set_xlabel("step")

axes[1].plot(ddpm_run["step_log"], ddpm_run["grad_norm_history"], label="lr=1e-3 (baseline)")
axes[1].plot(ddpm_lr_run["step_log"], ddpm_lr_run["grad_norm_history"], label=f"lr={BAD_LR} (failure 2)")
axes[1].legend(); axes[1].set_title("Gradient norm"); axes[1].set_xlabel("step"); axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

results["failure2_bad"] = {
    "lr": BAD_LR, "stopped_at_step": ddpm_lr_run["step_log"][-1], "diverged": ddpm_lr_run["diverged"],
    "max_grad_norm": float(np.nanmax(ddpm_lr_run["grad_norm_history"])),
    "baseline_max_grad_norm": float(np.nanmax(ddpm_run["grad_norm_history"])),
}


**Diagnosis:** with `lr=1.0` the Adam updates are large enough relative to the loss landscape that the
gradient norm plot above should show the excessive-LR run growing sharply compared to the `lr=1e-3` baseline,
and `train_ddpm`'s explicit `torch.isfinite(loss)` check (see `results["failure2_bad"]["diverged"]`) reports
whether the loss actually hit NaN/Inf and at which step training was halted. This is a standard
too-large-a-step instability: the optimizer overshoots the loss surface each step instead of descending it.

**Repair:** lower the learning rate back to `1e-3` (the working baseline from Section 3) — same architecture,
same schedule, same step count. The "post-repair" result is again the `ddpm_baseline` entry in `results`,
since Section 3 already is that exact configuration.


## 9. Auto-generated failure log

Built entirely from the `results` dict populated by the cells above — this is the one-page failure-and-fix
log deliverable. Run the notebook top to bottom, then copy the printed block below into the submission.


In [ ]:
def format_failure_log(results):
    lines = []
    lines.append("WEEK 4 FAILURE-AND-FIX LOG")
    lines.append("=" * 60)

    lines.append("\nFAILURE 1: Poor / mismatched DDPM noise schedule")
    f1 = results["failure1_bad"]
    b = results["ddpm_baseline"]
    lines.append(f"  Original setting: linear beta schedule, beta_start={f1['beta_start']}, "
                 f"beta_end={f1['beta_end']}, T={f1['T']} (vs. baseline cosine schedule, same T)")
    lines.append(f"  Observed symptom: training loss comparable to baseline "
                 f"({f1['final_train_loss']:.4f} vs {b['final_train_loss']:.4f}), but generated "
                 f"samples visibly off the two-moons manifold and MMD^2 much worse "
                 f"({f1['mmd2']:.5f} vs {b['mmd2']:.5f}); non-finite sample entries observed: "
                 f"{f1['n_nonfinite_samples']}")
    lines.append(f"  Evidence: alpha_bar drops below 0.01 by t={f1['collapse_step_bad']} under the "
                 f"mismatched schedule vs. t={f1['collapse_step_cos']} under cosine (both out of T={f1['T']}) "
                 f"-- {f1['frac_near_zero_snr_bad']:.0%} of the reverse trajectory runs at near-zero SNR under "
                 f"the mismatched schedule, vs. {f1['frac_near_zero_snr_cos']:.0%} under cosine. Training loss "
                 f"does not flag this (predicting eps from near-pure noise is an easy regression target).")
    lines.append(f"  Diagnosed cause: beta_end={f1['beta_end']} is far outside the standard DDPM range; "
                 f"it destroys signal too quickly rather than smoothly over the full schedule.")
    lines.append(f"  Repair: revert to the cosine (variance-preserving) schedule; no architecture or "
                 f"training-loop change required.")
    lines.append(f"  Result after repair: MMD^2={b['mmd2']:.5f} (baseline cosine run, Section 3), "
                 f"vs. {f1['mmd2']:.5f} under the mismatched schedule.")

    lines.append("\nFAILURE 2: Excessive learning rate")
    f2 = results["failure2_bad"]
    lines.append(f"  Original setting: lr={f2['lr']} (vs. working baseline lr=1e-3), same architecture, "
                 f"schedule, batch size, step budget")
    lines.append(f"  Observed symptom: training halted/diverged={f2['diverged']} "
                 f"(stopped logging at step {f2['stopped_at_step']} out of a 3000-step budget)")
    lines.append(f"  Evidence: gradient norm peaked at {f2['max_grad_norm']:.3e} vs. "
                 f"{f2['baseline_max_grad_norm']:.3e} for the lr=1e-3 baseline; "
                 f"loss non-finite check (torch.isfinite) triggered={f2['diverged']}")
    lines.append(f"  Diagnosed cause: lr=1.0 is roughly 1000x the working rate; Adam step size overshoots "
                 f"the loss surface instead of descending it, producing exploding gradients.")
    lines.append(f"  Repair: lower lr back to 1e-3 (all else unchanged).")
    lines.append(f"  Result after repair: final train loss={b['final_train_loss']:.4f}, "
                 f"MMD^2={b['mmd2']:.5f} (baseline cosine run, Section 3).")

    return "\n".join(lines)

failure_log_text = format_failure_log(results)
print(failure_log_text)

with open("failure_log.md", "w") as fh:
    fh.write(failure_log_text)
print("\nWritten to failure_log.md")


## 10. Final comparison and pilot recommendation

Auto-drafted from `results` — rerun the notebook and re-read this section rather than trusting any cached
text, since the wording below is generated from the actual numbers each time the cell runs.


In [ ]:
def format_recommendation(results):
    d, f, fno = results["ddpm_baseline"], results["flow_baseline"], results["fno"]
    lines = []
    lines.append("GENERATIVE TASK (DDPM vs. Rectified Flow, same two-moons dataset, same MMD^2 metric):")
    quality_winner = "DDPM" if d["mmd2"] < f["mmd2"] else "Rectified Flow"
    speed_winner = "DDPM" if f["sample_time_s"] > d["sample_time_s"] else "Rectified Flow"
    lines.append(f"  - Sample quality (MMD^2, lower better): DDPM={d['mmd2']:.5f}, Flow={f['mmd2']:.5f} "
                 f"-> {quality_winner} scored lower MMD^2 in this run.")
    lines.append(f"  - Sampling cost: DDPM={d['nfe']} NFEs / {d['sample_time_s']:.3f}s, "
                 f"Flow={f['nfe']} NFEs / {f['sample_time_s']:.3f}s "
                 f"-> {speed_winner} sampled faster in this run.")
    lines.append(f"  - Training cost: DDPM={d['train_time_s']:.2f}s / {d['peak_train_mem_mb']:.1f}MB peak, "
                 f"Flow={f['train_time_s']:.2f}s / {f['peak_train_mem_mb']:.1f}MB peak.")
    lines.append(f"  - Pilot pick for the generative task: weigh quality against the {speed_winner}'s lower "
                 f"NFE count if sampling latency matters for the pilot's use case; on a single free GPU, "
                 f"fewer NFEs at comparable quality is the practical win.")

    lines.append("\nPDE SURROGATE TASK (FNO) -- not ranked against the generative metric above (different task):")
    lines.append(f"  - Relative L2 @ training resolution ({fno['train_grid']} pts): "
                 f"{fno['rel_l2_train_res']:.4f} +/- {fno['rel_l2_train_res_std']:.4f}")
    lines.append(f"  - Relative L2 @ finer grid, zero-shot ({fno['fine_grid']} pts): "
                 f"{fno['rel_l2_fine_res']:.4f} +/- {fno['rel_l2_fine_res_std']:.4f}")
    degradation = fno["rel_l2_fine_res"] - fno["rel_l2_train_res"]
    lines.append(f"  - Measured change in relative L2 from train to finer resolution: {degradation:+.4f} "
                 f"(this is the actual transfer result -- not an assumption of discretization invariance).")
    lines.append(f"  - Training cost: {fno['train_time_s']:.2f}s / {fno['peak_train_mem_mb']:.1f}MB peak.")

    lines.append("\nSINGLE-FREE-GPU COMPUTE CONSTRAINT:")
    total_train_s = d["train_time_s"] + f["train_time_s"] + fno["train_time_s"]
    peak_mem_overall = max(d["peak_train_mem_mb"], f["peak_train_mem_mb"], fno["peak_train_mem_mb"])
    lines.append(f"  - Combined training wall-clock for all three models in this notebook: {total_train_s:.1f}s.")
    lines.append(f"  - Peak VRAM across the three training runs: {peak_mem_overall:.1f}MB "
                 f"(all comfortably inside a free-tier T4/Colab GPU at this toy scale).")

    lines.append("\nWHAT TO CHECK BEFORE SCALING:")
    lines.append("  - Re-run the MMD^2 comparison with a larger held-out reference set and multiple seeds; "
                 "a single 1000-point MMD estimate is noisy.")
    lines.append("  - For DDPM/flow: verify sample quality doesn't degrade if batch size or model width is "
                 "increased for a production dataset (this notebook only validates the toy 2D case).")
    lines.append("  - For FNO: test resolution transfer at a wider range of grid sizes and longer physical "
                 "time horizons than T_FINAL used here, and re-validate the finite-difference solver at any "
                 "new resolution before trusting its labels.")
    lines.append("  - Confirm wall-clock and VRAM scale acceptably if dataset size or model capacity grows "
                 "beyond this toy scale -- the numbers above are for the small toy-scale models only.")
    lines.append("  - Recheck numerical stability (the FTCS r<=0.5 condition) if alpha, dx, or dt change.")

    return "\n".join(lines)

recommendation_text = format_recommendation(results)
print(recommendation_text)

with open("recommendation.md", "w") as fh:
    fh.write(recommendation_text)
print("\nWritten to recommendation.md")


## 11. Rubric self-check (manual)

Go through this checklist after a full top-to-bottom run in Colab:

- [ ] **Diffusion and Flow Matching (20 pts):** Section 3 (DDPM) and Section 4 (Flow) both trained without
  `diverged=True` and produced sample scatter plots that visually resemble two moons.
- [ ] **Neural Operator (20 pts):** Section 6.2 solver validation assertion passed; Section 6.5 shows a
  measured (not assumed) relative L2 at both grid resolutions.
- [ ] **Failure Diagnosis (20 pts):** Sections 7 and 8 each show setting, symptom, evidence, diagnosis, repair,
  and post-repair result, all pulled from `results`.
- [ ] **Comparative Recommendation (20 pts):** Section 10 is generated from measured `results`, keeps the FNO
  relative L2 separate from the generative MMD^2, and names the single-GPU compute constraint.
- [ ] **Video Communication (20 pts):** record camera-on, 5-10 minutes, walking through the actual printed
  numbers and plots above — not this notebook's code.
